In [ ]:
import cv2
import json

# === Set image path ===
IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []

# Load the image
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height = frame.shape[0]  # used to flip Y-axis

# Mouse callback to collect L-shape points (flipped to math-style Y-axis)
def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        flipped_y = height - y  # flip Y-axis
        points.append((x, flipped_y))  # Store as math-style

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()

    # Draw selected points (convert back to OpenCV coordinate system for display)
    for point in points:
        draw_point = (point[0], height - point[1])  # Flip Y back for display
        cv2.circle(display_frame, draw_point, 5, (0, 0, 255), -1)

    if len(points) >= 2:
        pt1 = (points[0][0], height - points[0][1])
        pt2 = (points[1][0], height - points[1][1])
        cv2.line(display_frame, pt1, pt2, (0, 255, 0), 2)
    if len(points) == 3:
        pt2 = (points[1][0], height - points[1][1])
        pt3 = (points[2][0], height - points[2][1])
        cv2.line(display_frame, pt2, pt3, (0, 255, 0), 2)

    # Draw axis label
    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑", (10, height - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)  # Saved in math-style coords
        print(f"[✅] Saved L-shape to {OUTPUT_FILE}")
    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()


In [5]:
import os
import cv2
import json
import numpy as np
from ultralytics import YOLO

# === Load L-shape points from JSON ===
with open("lshape_image.json", "r") as f:
    l_points = json.load(f)

A, B, C = [np.array(pt, dtype=np.int32) for pt in l_points]
D = A + (C - B)

# === Load YOLO segmentation model ===
model = YOLO("tyre_seg/best.pt")  # Use your best.pt for tyre/truck segmentation
print("✅ Model loaded!")

# === Input image folder ===
input_folder = "test_images/wbtruck"
output_folder = "runs/segment/predict"

# === Process each image in the folder ===
for filename in os.listdir(input_folder):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    image_path = os.path.join(input_folder, filename)
    frame = cv2.imread(image_path)
    if frame is None:
        print(f"❌ Could not read {filename}")
        continue

    # === Run segmentation ===
    results = model.predict(source=frame, save=False)[0]

    # === Draw segmentation masks with color and contour ===
    if results.masks is not None:
        for i, mask in enumerate(results.masks.data):
            mask_np = mask.cpu().numpy()
            mask_resized = cv2.resize(mask_np, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)

            # Get class label for this mask
            cls_id = int(results.boxes.cls[i].item())
            label = model.names[cls_id].lower()

            # Set colors based on label
            if "truck" in label:
                color_fill = (0, 0, 255)       # Red fill
                color_border = (0, 0, 139)     # Dark red border
            elif "tyre" in label:
                color_fill = (255, 0, 0)       # Blue fill
                color_border = (255, 0, 0)     # Blue border
            else:
                # Skip unknown classes
                continue

            # Create colored mask
            colored_mask = np.zeros_like(frame)
            colored_mask[mask_resized > 0.5] = color_fill
            frame = cv2.addWeighted(frame, 1.0, colored_mask, 0.5, 0)

            # Find contours for border
            mask_uint8 = (mask_resized > 0.5).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(frame, contours, -1, color_border, 2)
    else:
        print(f"No masks detected in image: {filename}")

    # === Draw yellow corridor ===
    h, w = frame.shape[:2]
    overlay = frame.copy()
    vec_ab = B - A
    unit_ab = vec_ab / np.linalg.norm(vec_ab)
    perp = np.array([-unit_ab[1], unit_ab[0]])
    length = max(h, w) * 2
    A_ext = A - unit_ab * length
    B_ext = B + unit_ab * length
    C_ext = C + unit_ab * length
    D_ext = D - unit_ab * length
    main_corridor = np.array([A_ext, B_ext, C_ext, D_ext], dtype=np.int32)
    cv2.fillPoly(overlay, [main_corridor], (0, 255, 255))  # Yellow

    # === Blend overlay with base frame ===
    alpha = 0.3
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # === Save output image ===
    os.makedirs(output_folder, exist_ok=True)
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, frame)
    print(f"✅ Saved: {save_path}")


✅ Model loaded!

0: 640x480 (no detections), 495.5ms
Speed: 6.3ms preprocess, 495.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
No masks detected in image: t1.jpeg
✅ Saved: runs/segment/predict\t1.jpeg

0: 640x480 1 truck, 1 tyre, 425.1ms
Speed: 4.8ms preprocess, 425.1ms inference, 6.5ms postprocess per image at shape (1, 3, 640, 480)
✅ Saved: runs/segment/predict\t10.jpeg

0: 640x480 1 truck, 392.1ms
Speed: 5.1ms preprocess, 392.1ms inference, 3.9ms postprocess per image at shape (1, 3, 640, 480)
✅ Saved: runs/segment/predict\t11.jpeg

0: 640x480 1 truck, 1 tyre, 370.5ms
Speed: 3.9ms preprocess, 370.5ms inference, 6.2ms postprocess per image at shape (1, 3, 640, 480)
✅ Saved: runs/segment/predict\t12.jpeg

0: 640x480 2 tyres, 375.2ms
Speed: 4.7ms preprocess, 375.2ms inference, 7.3ms postprocess per image at shape (1, 3, 640, 480)
✅ Saved: runs/segment/predict\t13.jpeg

0: 640x480 1 truck, 1 tyre, 374.0ms
Speed: 4.1ms preprocess, 374.0ms inference, 6.2ms postproc

In [1]:
import cv2
import json

IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []

frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y  # math-style y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    # Draw horizontal grid lines (math Y)
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    
    # Draw vertical grid lines (X)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"  # vertical line
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    
    # Overlay math grid
    draw_math_grid(display_frame, spacing=50)

    # Draw points in math-style coordinates (convert for display)
    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    # Draw lines between points
    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)", 
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        print(f"📐 Line AB: {eq_ab}")
        print(f"📐 Line BC: {eq_bc}")

        # AB Alignment
        if A[1] == B[1]:
            print("🔷 AB is aligned with the X-axis (horizontal)")
        elif A[0] == B[0]:
            print("🔷 AB is aligned with the Y-axis (vertical)")
        else:
            print("🔷 AB is diagonal")

        # Area under AB to X-axis (trapezoid)
        x1, y1 = A
        x2, y2 = B
        area_ab = 0.5 * abs(x2 - x1) * (abs(y1) + abs(y2))
        print(f"📐 Area between AB and X-axis: {area_ab:.2f} pixels²")

        # Area under BC to Y-axis (trapezoid)
        x3, y3 = C
        area_bc = 0.5 * abs(y3 - y2) * (abs(x2) + abs(x3))
        print(f"📐 Area between BC and Y-axis: {area_bc:.2f} pixels²")

    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()


KeyboardInterrupt: 

In [1]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]

overlay_drawn = None  # Will hold the persistent overlay after save

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    # Apply persistent overlay if exists
    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        print(f"📐 Line AB: {eq_ab}")
        print(f"📐 Line BC: {eq_bc}")

        # AB region
        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            triangle_ab = np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (0, height)
            ], dtype=np.int32)
            area_ab = 0.5 * abs(Q[0] * P[1] - P[0] * Q[1])
            print(f"📍 AB intersects X at Q={Q}, Y at P={P}")
            print(f"📐 Area △OPQ: {area_ab:.2f} pixels²")
            cv2.fillPoly(overlay_drawn, [triangle_ab], (255, 0, 255))
            for pt, label in zip([Q, P, (0, 0)], ['Q', 'P', 'O']):
                cx, cy = int(pt[0]), int(pt[1])
                cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
                cv2.putText(overlay_drawn, label, (cx + 5, height - cy - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        # BC region
        if m_bc is not None:
            S = (-c_bc / m_bc, 0)
            R = (0, c_bc)
            triangle_bc = np.array([
                (int(S[0]), height - int(S[1])),
                (int(R[0]), height - int(R[1])),
                (0, height)
            ], dtype=np.int32)
            area_bc = 0.5 * abs(S[0] * R[1] - R[0] * S[1])
            print(f"📍 BC intersects X at S={S}, Y at R={R}")
            print(f"📐 Area △ORS: {area_bc:.2f} pixels²")
            cv2.fillPoly(overlay_drawn, [triangle_bc], (0, 255, 255))
            for pt, label in zip([S, R, (0, 0)], ['S', 'R', 'O']):
                cx, cy = int(pt[0]), int(pt[1])
                cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
                cv2.putText(overlay_drawn, label, (cx + 5, height - cy - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Saved L-shape points in math-style: [(227, 354), (442, 46), (854, 150)]
📐 Line AB: y = -1.43x + 679.19
📐 Line BC: y = 0.25x + -65.57
📍 AB intersects X at Q=(474.1103896103896, 0), Y at P=(0, 679.1906976744185)
📐 Area △OPQ: 161005.68 pixels²
📍 BC intersects X at S=(259.7692307692308, 0), Y at R=(0, -65.57281553398059)
📐 Area △ORS: 8516.90 pixels²


In [1]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        # --- AB triangle: O-P-Q ---
        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)

            triangle_ab = np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ], dtype=np.int32)

            cv2.fillPoly(overlay_drawn, [triangle_ab], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                cx, cy = int(pt[0]), int(pt[1])
                cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
                cv2.putText(overlay_drawn, label, (cx + 5, height - cy - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        # --- BC triangle: B-C-R ---
        if m_bc is not None:
            max_x = width
            y_at_max_x = m_bc * max_x + c_bc
            R = (max_x, y_at_max_x)

            triangle_bc = np.array([
                (int(B[0]), height - int(B[1])),
                (int(C[0]), height - int(C[1])),
                (int(R[0]), height - int(R[1]))
            ], dtype=np.int32)

            cv2.fillPoly(overlay_drawn, [triangle_bc], (0, 255, 255))

            for pt, label in zip([B, C, R], ['B', 'C', 'R']):
                cx, cy = int(pt[0]), int(pt[1])
                cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
                cv2.putText(overlay_drawn, label, (cx + 5, height - cy - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

            # --- Point S: Extend BC to X-axis (y=0) ---
            if m_bc != 0:
                x_s = -c_bc / m_bc
                S = (x_s, 0)
                if 0 <= x_s <= width:
                    cx, cy = int(S[0]), int(S[1])
                    cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
                    cv2.putText(overlay_drawn, "S", (cx + 5, height - cy - 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
                    print(f"📍 Line BC intersects X-axis at S={S}")

        # --- Draw A, B, C explicitly ---
        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            cx, cy = int(pt[0]), int(pt[1])
            cv2.circle(overlay_drawn, (cx, height - cy), 5, (0, 0, 0), -1)
            cv2.putText(overlay_drawn, label, (cx + 5, height - cy - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Saved L-shape points in math-style: [(229, 344), (450, 28), (852, 136)]
📍 Line BC intersects X-axis at S=(345.77777777777777, 0)


In [3]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, _ = line_equation(A, B)
        m_bc, c_bc, _ = line_equation(B, C)

        # AB Triangle (OPQ)
        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)
            cv2.fillPoly(overlay_drawn, [np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ])], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                draw_point_label(overlay_drawn, pt, label)

        # Extended BC line intersection points
        if m_bc is not None:
            x_left, y_left = 0, c_bc
            x_right, y_right = width, m_bc * width + c_bc

            R = (x_right, y_right)
            S = (x_left, y_left)
            draw_point_label(overlay_drawn, R, "R")
            draw_point_label(overlay_drawn, S, "S")

            # Shade full region between BC line and bottom X-axis
            bottom_right = (width, 0)
            bottom_left = (0, 0)

            extended_bc_area = np.array([
                (int(S[0]), height - int(S[1])),
                (int(R[0]), height - int(R[1])),
                (bottom_right[0], height - bottom_right[1]),
                (bottom_left[0], height - bottom_left[1])
            ], dtype=np.int32)

            cv2.fillPoly(overlay_drawn, [extended_bc_area], (0, 200, 200))

        # Mark A, B, C
        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            draw_point_label(overlay_drawn, pt, label)

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Saved L-shape points in math-style: [(224, 354), (445, 30), (857, 140)]


In [5]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/t3.jpeg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        print("\n==============================")
        print("L-SHAPE ROI COORDINATE SUMMARY")
        print("==============================")
        print(f"A = {A}")
        print(f"B = {B}")
        print(f"C = {C}")

        print("\n--- Line AB ---")
        print(f"Equation of AB: {eq_ab}")

        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)
            print(f"Q = {Q}  (X-intercept of AB)")
            print(f"P = {P}  (Y-intercept of AB)")
            print(f"O = {O}  (Origin)")
            cv2.fillPoly(overlay_drawn, [np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ])], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                draw_point_label(overlay_drawn, pt, label)
        else:
            print("Line AB is vertical. No standard slope-intercept form.")

        print("\n--- Line BC ---")
        print(f"Equation of BC: {eq_bc}")

        if m_bc is not None:
            S = (0, c_bc)
            R = (width, m_bc * width + c_bc)
            print(f"S = {S}  (Left intersection of extended BC)")
            print(f"R = {R}  (Right intersection of extended BC)")
        else:
            print("Line BC is vertical. No standard slope.")
            S = (B[0], 0)
            R = (B[0], height)

        bottom_left = (0, 0)
        bottom_right = (width, 0)

        print("\n--- Shaded Region Points (Extended BC to Bottom) ---")
        print(f"1. S = {S}")
        print(f"2. R = {R}")
        print(f"3. Bottom Right = {bottom_right}")
        print(f"4. Bottom Left = {bottom_left}")
        print("==============================\n")

        extended_bc_area = np.array([
            (int(S[0]), height - int(S[1])),
            (int(R[0]), height - int(R[1])),
            (bottom_right[0], height - bottom_right[1]),
            (bottom_left[0], height - bottom_left[1])
        ], dtype=np.int32)
        cv2.fillPoly(overlay_drawn, [extended_bc_area], (0, 200, 200))

        # Mark A, B, C
        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            draw_point_label(overlay_drawn, pt, label)

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Saved L-shape points in math-style: [(226, 346), (442, 30), (850, 144)]

L-SHAPE ROI COORDINATE SUMMARY
A = (226, 346)
B = (442, 30)
C = (850, 144)

--- Line AB ---
Equation of AB: y = -1.46x + 676.63
Q = (462.506329113924, 0)  (X-intercept of AB)
P = (0, 676.6296296296296)  (Y-intercept of AB)
O = (0, 0)  (Origin)

--- Line BC ---
Equation of BC: y = 0.28x + -93.50
S = (0, -93.5)  (Left intersection of extended BC)
R = (1200, 241.79411764705884)  (Right intersection of extended BC)

--- Shaded Region Points (Extended BC to Bottom) ---
1. S = (0, -93.5)
2. R = (1200, 241.79411764705884)
3. Bottom Right = (1200, 0)
4. Bottom Left = (0, 0)



In [ ]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/tp.jpg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape points in math-style: {points}")

        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        print("\n==============================")
        print("L-SHAPE ROI COORDINATE SUMMARY")
        print("==============================")
        print(f"A = {A}")
        print(f"B = {B}")
        print(f"C = {C}")

        print("\n--- Line AB ---")
        print(f"Equation of AB: {eq_ab}")

        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)
            print(f"Q = {Q}  (X-intercept of AB)")
            print(f"P = {P}  (Y-intercept of AB)")
            print(f"O = {O}  (Origin)")
            cv2.fillPoly(overlay_drawn, [np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ])], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                draw_point_label(overlay_drawn, pt, label)
        else:
            print("Line AB is vertical. No standard slope-intercept form.")

        print("\n--- Line BC ---")
        print(f"Equation of BC: {eq_bc}")

        if m_bc is not None:
            S = (0, c_bc)
            R = (width, m_bc * width + c_bc)
            print(f"S = {S}  (Left intersection of extended BC)")
            print(f"R = {R}  (Right intersection of extended BC)")
        else:
            print("Line BC is vertical. No standard slope.")
            S = (B[0], 0)
            R = (B[0], height)

        bottom_left = (0, 0)
        bottom_right = (width, 0)

        print("\n--- Shaded Region Points (Extended BC to Bottom) ---")
        print(f"1. S = {S}")
        print(f"2. R = {R}")
        print(f"3. Bottom Right = {bottom_right}")
        print(f"4. Bottom Left = {bottom_left}")
        print("==============================\n")

        extended_bc_area = np.array([
            (int(S[0]), height - int(S[1])),
            (int(R[0]), height - int(R[1])),
            (bottom_right[0], height - bottom_right[1]),
            (bottom_left[0], height - bottom_left[1])
        ], dtype=np.int32)
        cv2.fillPoly(overlay_drawn, [extended_bc_area], (0, 200, 200))

        # Mark A, B, C
        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            draw_point_label(overlay_drawn, pt, label)

        # ✅ Check point K = (25, 35)
        K = (400, 350)
        K_cv = (int(K[0]), height - int(K[1]))  # Convert to OpenCV coordinate system

        triangle_pts = np.array([
            (int(Q[0]), height - int(Q[1])),
            (int(P[0]), height - int(P[1])),
            (int(O[0]), height - int(O[1]))
        ], dtype=np.int32)

        bc_area_pts = np.array([
            (int(S[0]), height - int(S[1])),
            (int(R[0]), height - int(R[1])),
            (bottom_right[0], height - bottom_right[1]),
            (bottom_left[0], height - bottom_left[1])
        ], dtype=np.int32)

        in_triangle = cv2.pointPolygonTest(triangle_pts, K_cv, False) >= 0
        in_bc_area = cv2.pointPolygonTest(bc_area_pts, K_cv, False) >= 0

        if in_triangle or in_bc_area:
            print(f"K = {K} is OUTSIDE (inside or on the shaded region)")
            cv2.circle(overlay_drawn, K_cv, 6, (0, 0, 255), -1)
            cv2.putText(overlay_drawn, "K (OUTSIDE)", (K_cv[0] + 5, K_cv[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        else:
            print(f"K = {K} is INSIDE (not in any shaded region)")
            cv2.circle(overlay_drawn, K_cv, 6, (0, 255, 0), -1)
            cv2.putText(overlay_drawn, "K (INSIDE)", (K_cv[0] + 5, K_cv[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


In [2]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/tp.jpg"
OUTPUT_FILE = "lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def to_list(pt):
    return [int(pt[0]), int(pt[1])]

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        Q = P = O = None
        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)
            triangle_pts = np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ], dtype=np.int32)
            cv2.fillPoly(overlay_drawn, [triangle_pts], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                draw_point_label(overlay_drawn, pt, label)

        if m_bc is not None:
            S = (0, c_bc)
            R = (width, m_bc * width + c_bc)
        else:
            S = (B[0], 0)
            R = (B[0], height)

        bottom_left = (0, 0)
        bottom_right = (width, 0)

        extended_bc_area = np.array([
            (int(S[0]), height - int(S[1])),
            (int(R[0]), height - int(R[1])),
            (bottom_right[0], height - bottom_right[1]),
            (bottom_left[0], height - bottom_left[1])
        ], dtype=np.int32)
        cv2.fillPoly(overlay_drawn, [extended_bc_area], (0, 200, 200))

        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            draw_point_label(overlay_drawn, pt, label)
        for pt, label in zip([S, R, bottom_right, bottom_left], ['S', 'R', 'BR', 'BL']):
            draw_point_label(overlay_drawn, pt, label)

        # Save all data (convert all to lists)
        save_data = {
            "A": to_list(A),
            "B": to_list(B),
            "C": to_list(C),
            "line_AB": eq_ab,
            "line_BC": eq_bc,
            "Q": to_list(Q) if Q else None,
            "P": to_list(P) if P else None,
            "O": to_list(O) if O else None,
            "S": to_list(S),
            "R": to_list(R),
            "BOTTOM_RIGHT": to_list(bottom_right),
            "BOTTOM_LEFT": to_list(bottom_left)
        }

        with open(OUTPUT_FILE, 'w') as f:
            json.dump(save_data, f, indent=2)

        print(f"[✅] Extended L-shape data saved to: {OUTPUT_FILE}")

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Extended L-shape data saved to: lshape_image.json


In [2]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/wbtruck/t1.jpeg"
L_SHAPE_FILE = "lshape_image.json"

frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]

# === Load L-shape data from JSON ===
with open(L_SHAPE_FILE, "r") as f:
    data = json.load(f)

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 255), -1)
        cv2.putText(img, label, (cx + 5, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

overlay = np.zeros_like(frame)

# === Recover points ===
def to_np(pt): return np.array(pt, dtype=np.int32)
A = to_np(data["A"])
B = to_np(data["B"])
C = to_np(data["C"])
Q = to_np(data["Q"]) if data["Q"] else None
P = to_np(data["P"]) if data["P"] else None
O = to_np(data["O"]) if data["O"] else None
S = to_np(data["S"])
R = to_np(data["R"])
BOTTOM_RIGHT = to_np(data["BOTTOM_RIGHT"])
BOTTOM_LEFT = to_np(data["BOTTOM_LEFT"])

# === Draw AB and BC lines ===
cv2.line(overlay, (A[0], height - A[1]), (B[0], height - B[1]), (0, 255, 0), 2)
cv2.line(overlay, (B[0], height - B[1]), (C[0], height - C[1]), (0, 255, 0), 2)

# === Draw labels ===
for pt, label in zip([A, B, C, S, R, BOTTOM_RIGHT, BOTTOM_LEFT], ['A', 'B', 'C', 'S', 'R', 'BR', 'BL']):
    draw_point_label(overlay, pt, label)

if Q is not None and P is not None and O is not None:
    for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
        draw_point_label(overlay, pt, label)
    triangle = np.array([
        (Q[0], height - Q[1]),
        (P[0], height - P[1]),
        (O[0], height - O[1])
    ])
    cv2.fillPoly(overlay, [triangle], (255, 0, 255))

# === Draw extended BC region ===
bc_shade = np.array([
    (S[0], height - S[1]),
    (R[0], height - R[1]),
    (BOTTOM_RIGHT[0], height - BOTTOM_RIGHT[1]),
    (BOTTOM_LEFT[0], height - BOTTOM_LEFT[1])
])
cv2.fillPoly(overlay, [bc_shade], (0, 200, 200))

# === Display result ===
final = cv2.addWeighted(frame, 1.0, overlay, 0.6, 0)
cv2.namedWindow("L-shape ROI Auto Draw", cv2.WINDOW_NORMAL)
cv2.imshow("L-shape ROI Auto Draw", final)
cv2.waitKey(0)
cv2.destroyAllWindows()
